In [1]:
from idlelib.debugobj import dispatch

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import torch
import matplotlib as mpl
from pandas.io.sas.sas_constants import truncated_subheader_id
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import warnings
from torch import  nn
from torch.utils.data import DataLoader
import joblib

from sklearn.pipeline import make_pipeline

warnings.filterwarnings('ignore')


seed = 67

colors = ["#009ED0","#c3c8be","#f2afcb"]


cm = mpl.colors.LinearSegmentedColormap.from_list('colorcitos',colors)



inmutable_df = pd.read_parquet('../data/processed/CLASF_short_data.parquet')

In [2]:
df = inmutable_df.copy()
df = df.reset_index().drop(columns=['index'])
df


,rating,y,w1,w2,w3,w4,w5,w6,w7,w8,...,w759,w760,w761,w762,w763,w764,w765,w766,w767,w768
0,5,gift_cards,-0.024887,0.007899,0.003009,-0.094055,-0.003241,-0.010345,-0.032938,0.025727,...,0.003551,-0.046339,-0.041232,-0.013794,-0.049453,-0.011040,-0.010916,0.034731,0.046359,-0.024646
1,5,digital_music,-0.028520,-0.003986,-0.002436,-0.119818,0.033662,0.002148,-0.031996,0.038514,...,0.050848,-0.014365,0.013234,-0.021985,-0.021832,0.007104,0.013809,0.058802,-0.020029,-0.020273
2,5,gift_cards,0.005387,-0.003584,-0.010338,-0.149245,-0.009772,0.007169,0.015875,-0.000210,...,0.015273,-0.008234,-0.023015,-0.025498,0.005176,-0.010098,0.030468,0.035991,0.011597,-0.038796
3,1,digital_music,-0.020638,-0.052467,-0.014607,-0.099339,0.031590,-0.013430,-0.018903,0.011315,...,0.010960,0.022318,-0.053241,-0.056295,-0.030914,-0.014435,0.016317,0.007946,0.074728,0.033262
4,3,magazines,-0.027466,-0.044827,0.004275,-0.053112,-0.044739,-0.018625,-0.000894,0.030281,...,-0.016350,-0.048549,0.013798,-0.005266,-0.018492,0.027676,0.055425,0.003952,0.028558,0.032652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,5,gift_cards,-0.048660,-0.014467,-0.016132,-0.116516,0.026375,-0.001256,-0.020911,0.035611,...,0.023353,-0.033935,-0.018241,-0.022677,-0.017755,0.000983,-0.007757,0.044418,0.013258,0.004044
119996,5,gift_cards,-0.009364,0.017477,-0.003649,-0.101146,0.002632,-0.017567,0.051485,0.019261,...,0.016276,0.016702,0.003037,-0.015529,0.037199,0.015329,0.003755,0.027804,0.003915,0.003337
119997,5,gift_cards,-0.011336,-0.012354,0.013837,-0.106397,0.029282,0.019965,-0.000988,0.013854,...,0.041929,-0.052962,-0.015344,-0.037097,-0.007420,0.002436,0.023683,0.004912,-0.013263,-0.020518
119998,5,magazines,-0.040870,-0.049382,0.013034,-0.105640,-0.030041,-0.016332,-0.016006,-0.005308,...,0.004140,-0.014351,-0.024094,-0.051654,-0.029038,0.020465,0.007640,0.073231,-0.001746,0.025817


In [3]:

Y = df['y'].copy()
X = df.drop(columns=['y'])


In [4]:

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=seed,stratify=Y)
X_tunning,X_val,Y_tunnning,Y_val = train_test_split(X_train,Y_train,test_size=0.2,random_state=seed,stratify=Y_train)


# Propocision de modelos

Desde que vimos que hay grupos mas densos en unas regiones que en otros vamos a utilizar KNN


- KNN




In [5]:
def knn_optuna_objective(trial):
    print('lo primero')
    k_n = trial.suggest_int('knn_n_neighbors',2,20,step=2)
    p = trial.suggest_int('p',1,6)
    metric = trial.suggest_categorical('metric',['minkowski','cosine'])

    print('segundo')
    knn = KNeighborsClassifier(n_neighbors=k_n,p=p,metric=metric,n_jobs=1,algorithm='brute')
    print('tercero')
    knn.fit(X_tunning,Y_tunnning)

    y_pred = knn.predict(X_val)

    print('cuarto')

    return f1_score(y_pred=y_pred,y_true=Y_val,average='weighted')



In [6]:
# stop here apartir de aqui se pone tardado el asunto

SyntaxError: invalid syntax (1116318007.py, line 1)

In [6]:
knn_study = optuna.create_study(direction='maximize')
knn_study.optimize(knn_optuna_objective,n_trials=12)

[I 2025-11-20 00:42:16,343] A new study created in memory with name: no-name-a085e6e3-1a29-4246-8863-1e0ebcf16651


lo primero
segundo
tercero


[I 2025-11-20 00:42:41,370] Trial 0 finished with value: 0.8947071519876674 and parameters: {'knn_n_neighbors': 20, 'p': 1, 'metric': 'cosine'}. Best is trial 0 with value: 0.8947071519876674.


cuarto
lo primero
segundo
tercero


[I 2025-11-20 00:43:04,392] Trial 1 finished with value: 0.896461065167397 and parameters: {'knn_n_neighbors': 12, 'p': 5, 'metric': 'cosine'}. Best is trial 1 with value: 0.896461065167397.


...

cuarto
lo primero
segundo
tercero


[I 2025-11-20 00:46:10,044] Trial 10 finished with value: 0.874738879779785 and parameters: {'knn_n_neighbors': 2, 'p': 2, 'metric': 'minkowski'}. Best is trial 9 with value: 0.9005120648051543.


cuarto
lo primero
segundo
tercero


[I 2025-11-20 01:26:39,731] Trial 11 finished with value: 0.8972742139964527 and parameters: {'knn_n_neighbors': 14, 'p': 6, 'metric': 'minkowski'}. Best is trial 9 with value: 0.9005120648051543.


cuarto


In [7]:
knn_study.best_params

{'knn_n_neighbors': 16, 'p': 2, 'metric': 'minkowski'}

In [13]:
knn_study.best_trial

FrozenTrial(number=9, state=1, values=[0.9005120648051543], datetime_start=datetime.datetime(2025, 11, 20, 0, 45, 48, 378756), datetime_complete=datetime.datetime(2025, 11, 20, 0, 45, 59, 175669), params={'knn_n_neighbors': 16, 'p': 2, 'metric': 'minkowski'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'knn_n_neighbors': IntDistribution(high=20, log=False, low=2, step=2), 'p': IntDistribution(high=6, log=False, low=1, step=1), 'metric': CategoricalDistribution(choices=('minkowski', 'cosine'))}, trial_id=9, value=None)

In [7]:


knn_model = make_pipeline(KNeighborsClassifier(n_neighbors=16,p=2,metric='minkowski'))
knn_model.fit(X_train,Y_train)
y_pred = knn_model.predict(X_test)
print(classification_report(y_true=Y_test,y_pred=y_pred))



                    precision    recall  f1-score   support

     digital_music       0.94      0.86      0.90      8499
        gift_cards       0.85      0.96      0.90      9832
         magazines       0.92      0.86      0.89      4624
subscription_boxes       0.92      0.73      0.82      1045

          accuracy                           0.89     24000
         macro avg       0.91      0.85      0.88     24000
      weighted avg       0.90      0.89      0.89     24000



### KNN

#### Logistic regresion

In [8]:


log_reg_model = make_pipeline(LogisticRegression(random_state=seed))
log_reg_model.fit(X_train,Y_train)
y_pred = log_reg_model.predict(X_test)



print(classification_report(y_pred=y_pred,y_true=Y_test))


                    precision    recall  f1-score   support

     digital_music       0.92      0.89      0.90      8499
        gift_cards       0.87      0.95      0.91      9832
         magazines       0.93      0.86      0.89      4624
subscription_boxes       0.92      0.75      0.82      1045

          accuracy                           0.90     24000
         macro avg       0.91      0.86      0.88     24000
      weighted avg       0.90      0.90      0.90     24000



# Random Forest

quisimos probar Random Forest por su robustez ante datos nuevos y su proteccion ante datos atipicos, tiene ademas la virtud de que  nos dice las variables mas importantes pero en este caso por el embedding hasta lo que se hacer ahora mismo no nos es de muchisima utilidad

In [17]:

def rf_optuna_objective(trial):
    n_estimators = trial.suggest_int('n_estimators',60,200,step=25)
    criterion = trial.suggest_categorical('criterion',['gini','entropy','log_loss'])
    max_depth = trial.suggest_int('max_depht',20,150,step=10)
    min_sample_split = trial.suggest_int('min_sample_split',2,16,step=2)

    rf = RandomForestClassifier(n_estimators=n_estimators,criterion=criterion,max_depth=max_depth,min_samples_split=min_sample_split,bootstrap=True,n_jobs=-1)
    rf.fit(X_tunning,Y_tunnning)

    y_pred = rf.predict(X_val)

    return f1_score(y_pred=y_pred,y_true=Y_val,average='weighted')



In [21]:

rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_optuna_objective,n_trials=22)

[I 2025-11-20 19:40:11,605] A new study created in memory with name: no-name-1f58c4cd-ebd6-41b6-89f4-0f6df20f2298
[I 2025-11-20 19:40:35,445] Trial 0 finished with value: 0.8659581193543113 and parameters: {'n_estimators': 60, 'criterion': 'log_loss', 'max_depht': 70, 'min_sample_split': 4}. Best is trial 0 with value: 0.8659581193543113.
[I 2025-11-20 19:41:17,054] Trial 1 finished with value: 0.8660278142272992 and parameters: {'n_estimators': 110, 'criterion': 'gini', 'max_depht': 70, 'min_sample_split': 4}. Best is trial 1 with value: 0.8660278142272992.
[I 2025-11-20 19:41:58,020] Trial 2 finished with value: 0.8667394048495525 and parameters: {'n_estimators': 110, 'criterion': 'gini', 'max_depht': 140, 'min_sample_split': 12}. Best is trial 2 with value: 0.8667394048495525.
[I 2025-11-20 19:43:01,816] Trial 3 finished with value: 0.8718669297871694 and parameters: {'n_estimators': 185, 'criterion': 'log_loss', 'max_depht': 130, 'min_sample_split': 6}. Best is trial 3 with value: 

In [22]:
rf_study.best_params

{'n_estimators': 160,
 'criterion': 'entropy',
 'max_depht': 40,
 'min_sample_split': 6}

In [9]:
rf_study_best_params = {'n_estimators': 160,
 'criterion': 'entropy',
 'max_depth': 40,
 'min_samples_split': 6}

In [10]:
rf_model = make_pipeline(RandomForestClassifier(**rf_study_best_params,random_state=seed))
rf_model.fit(X_train,Y_train)
y_pred = rf_model.predict(X_test)
print(classification_report(y_pred=y_pred,y_true=Y_test))

                    precision    recall  f1-score   support

     digital_music       0.89      0.88      0.89      8499
        gift_cards       0.83      0.95      0.88      9832
         magazines       0.94      0.79      0.86      4624
subscription_boxes       0.98      0.40      0.57      1045

          accuracy                           0.87     24000
         macro avg       0.91      0.76      0.80     24000
      weighted avg       0.88      0.87      0.87     24000



In [ ]:

# guardamos  los modelos

joblib.dump(knn_model,'../models/knn.pkl')

In [13]:
joblib.dump(log_reg_model,'../models/log_reg.pkl')

['../models/log_reg.pkl']

In [14]:

joblib.dump(rf_model,'../models/rf.pkl')


['../models/rf.pkl']

MLP

veamos de cuantas neuronas nos toca la capa de entrada, de forma
adicional sabemos de antemano que nuestras clases son mutuamente excluyentes,
esto es que no pueden ocurrir dos al mismo tiempo, debido debemos de usar la
_softmax_ en nuestra ultima capa

In [7]:
# codificamos y
mappings = {k:v for v,k in enumerate(Y_train.unique())}

In [8]:
n_input_layer = X_train.shape[1]
n_output_layer = Y_train.nunique()
display(
    n_input_layer,
    n_output_layer
)

769

4

In [9]:
# wrappeando la data para que pytorch pueda trabajar con ella

df_train = pd.concat([X_train,Y_train.map(mappings)],axis=1)
df_test = pd.concat([X_test,Y_test.map(mappings)],axis=1)


#train_data = DataLoader(df_train)
#test_data = DataLoader(df_test)


class TorchData:
    def __init__(self,X,Y):
        self.X = X
        self.Y = Y

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.Y[idx]
        x = torch.tensor(x,dtype=torch.float32)
        y = torch.tensor(y,dtype=torch.long)
        return x,y
    def __len__(self):
        return len(self.X)


train_data = TorchData(X_train.values,Y_train.map(mappings).values)
test_data = TorchData(X_test.values,Y_test.map(mappings).values)

train_loader = DataLoader(train_data,batch_size=64,shuffle=True,num_workers=1)
test_loader = DataLoader(test_data,batch_size=64,shuffle=True,num_workers=1)



In [10]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
# MLP

class MlpClassifier(nn.Module):
    # constructor de la red
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() # No se si es realmente necesario esta linea
        self.layers = nn.Sequential(

            nn.Linear(n_input_layer, 256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, n_output_layer)

        )


    def forward(self,x):
        x = self.flatten(x)
        return self.layers(x)


model = MlpClassifier().to(device)
print(model)






Using cpu device
MlpClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layers): Sequential(
    (0): Linear(in_features=769, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [11]:

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [12]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [13]:
epochs = 30
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    test(test_loader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.377234  [   64/96000]
loss: 0.266830  [ 6464/96000]
loss: 0.392140  [12864/96000]
loss: 0.201658  [19264/96000]
loss: 0.283553  [25664/96000]
loss: 0.220915  [32064/96000]
loss: 0.149094  [38464/96000]
loss: 0.367531  [44864/96000]
loss: 0.210578  [51264/96000]
loss: 0.164600  [57664/96000]
loss: 0.212916  [64064/96000]
loss: 0.242840  [70464/96000]
loss: 0.181049  [76864/96000]
loss: 0.395942  [83264/96000]
loss: 0.298942  [89664/96000]
Test Error: 
 Accuracy: 89.9%, Avg loss: 0.256449 

Epoch 2
-------------------------------
loss: 0.346177  [   64/96000]
loss: 0.173368  [ 6464/96000]
loss: 0.249703  [12864/96000]
loss: 0.305662  [19264/96000]
loss: 0.221913  [25664/96000]
loss: 0.236357  [32064/96000]
loss: 0.217678  [38464/96000]
loss: 0.409708  [44864/96000]
loss: 0.314302  [51264/96000]
loss: 0.184459  [57664/96000]
loss: 0.190931  [64064/96000]
loss: 0.293972  [70464/96000]
loss: 0.272374  [76864/96000]
loss: 0.216128  [83264/96000

In [14]:
model.eval()

MlpClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layers): Sequential(
    (0): Linear(in_features=769, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=4, bias=True)
  )
)

In [15]:
# Hacer predicciones
model.eval()
preds = []

with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        outputs = model(x)
        pred = torch.argmax(outputs, dim=1)
        preds.extend(pred.cpu().numpy())

In [16]:
print(classification_report(y_pred=preds,y_true=Y_test.map(mappings)))

              precision    recall  f1-score   support

           0       0.41      0.45      0.43      9832
           1       0.19      0.18      0.18      4624
           2       0.36      0.34      0.35      8499
           3       0.04      0.04      0.04      1045

    accuracy                           0.34     24000
   macro avg       0.25      0.25      0.25     24000
weighted avg       0.33      0.34      0.34     24000



# Conclusiones

Waos esto si no me lo esperaba el MLP ha sido el modelo que peor se comporta con diferencia, pytorch es un framework que pese a mas intuitivo que tensorflow, requiere muchos pasos adicionales en los que me pude haber equivocado, ademas de que el tuneo de hiperparametros es mas complicado, en este caso lo que hice fue probar diferentes configuraciones de optimizers, quedandome al final con Adam, la verdad es que no se entonces porque al estar haciendo los epochs se muestra un accuracy de 90% lo que hice fue simplemente tomar las funciones genericas que estan en la documentacion de pytorch y entrenar la red, no me sorprenderia que hubiera cometido un error sacando las metricas. con el resto de los modelos la verdad es que se comportaron bastante bien, ya mencionamos que aqui no hay falsos positivos mas costosos que falsos negativos y viceversa, aunque si hay un desvalance normal de las clases, por lo que nos decidimos quedar con la _regresion logistica_ que presenta _f1_score= .90_ seguido por el KNN el Random Forest y por ultimo la red neuronal, algo contrario a lo que habriamos pensado al principio cuando deciamos que los clusteres eran difusos y que iba a ser dificil separar los datos, resulta que los modelos mas sencillos y vagos son los que mejor se comportaron, esto evidentemente con las limitantes que habia quizas con el tuneo de las redes. Por lo tanto vamos a hacer la prediccion del resto de los datos con la regresion logistica

# Testeos

In [4]:

df_long = pd.read_parquet('../data/processed/CLASF_long_data.parquet')
log_reg = joblib.load('../models/log_reg.pkl')
Y = df_long['y']
X = df_long.drop(columns=['y'])

In [7]:
y_pred = log_reg.predict(X)
print(classification_report(y_pred=y_pred,y_true=Y))

                    precision    recall  f1-score   support

     digital_music       0.92      0.89      0.90     86655
        gift_cards       0.87      0.95      0.91    101821
         magazines       0.94      0.86      0.90     47625
subscription_boxes       0.91      0.74      0.82     10788

          accuracy                           0.90    246889
         macro avg       0.91      0.86      0.88    246889
      weighted avg       0.90      0.90      0.90    246889



Los resultados son buenos con los otros 240_000 datos  que no usamos

#### Referencia
- https://github.com/GLAZERadr/Multi-Layer-Perceptron-Pytorch/blob/main/model/MultiLayerPerceptron.ipynb
- https://stackoverflow.com/questions/56107259/how-to-save-a-trained-model-by-scikit-learn
- https://docs.pytorch.org/tutorials/recipes/recipes/defining_a_neural_network.html
- https://optuna.org/#code_examples